# 24 · Stage 3 — train student trên nhãn đã sửa, so với `H0`

Đọc `Y_HARD / M_HARD / Y_SOFT / M_SOFT` do `21_label_pipeline_v1.ipynb` sinh ra cho một arm, rồi
train **đúng setup R0 của nb16** với loss

\[ L=\frac{\sum M^{hard}L_{ASL}(Y^{hard},z)+\lambda_{soft}\sum M^{soft}L_{BCE}(Y^{soft},z)}{B_{actual}\times 13} \]

Khác nb16 **duy nhất ở kênh nhãn**. Kiến trúc, split, preprocessing, augmentation, view pooling,
LR/scheduler, epoch, checkpoint rule đều giữ nguyên.

`H0` = arm `R0` seed 42 của nb16 — đã train sẵn, đọc lại từ `label_reliability/`, **không train lại**.

| cell | compute | nội dung |
|---|---|---|
| C0–C1 | CPU | config · cohort · ma trận nhãn của arm · VAL historical |
| C2–C3 | GPU | stage ảnh · dataset · model R0 · `student_loss` |
| C4 | CPU | bộ đo + checkpoint rule §9 |
| C5 | **GPU ~6 h/seed** | train 8 epoch, eval VAL mỗi epoch, resume theo epoch |
| C6–C7 | CPU | so với `H0` · cổng §14 · bootstrap subject-cluster |

## C0 · Config

In [ ]:
import os, re, json, math, time, copy, pickle, random, shutil, tarfile, hashlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')
LP1_ROOT  = os.path.join(DATA_DIR, 'label_pipeline_v1')
SC16      = os.path.join(DATA_DIR, 'label_reliability')          # nb16: baseline H0, CHI DOC

# ================= CHON ARM + SEED =================
ARM_TAG = 'A2E_full'        # <<< 'A2_allzero' | 'A3_negshare' | 'A0_allone' | 'A1_allsoft'
SEED    = 42                  # <<< 42 truoc; 1337 / 2024 de xac nhan (§11)
LP1     = os.path.join(LP1_ROOT, ARM_TAG)
STU     = os.path.join(LP1, f'student_s{SEED}')
for _d in (STU, f'{STU}/preds', f'{STU}/snapshots', f'{STU}/metrics'):
    os.makedirs(_d, exist_ok=True)
assert os.path.exists(f'{LP1}/M_HARD.npy'), (
    f'chua co ma tran nhan cho arm {ARM_TAG}. Chay 21_label_pipeline_v1.ipynb voi '
    f"BLANK_ROUTE_MODE tuong ung truoc.")

# ================= KHOA: y het R0 nb16 =================
EPOCHS, EFF = 8, 256
LR_BLOCKS, LR_HEAD = 1e-5, 1e-5
CLIP_BLOCKS, CLIP_HEAD = 5.0, 1.0
WD, WARMUP_FRAC, HEAD_WARMUP_EPOCHS = 1e-2, 0.05, 1
ASL_GP, ASL_GN, ASL_CLIP, ASL_EPS = 1, 4, 0.05, 1e-8
LAMBDA_SOFT = 0.5
W_F, W_L = 0.7, 0.3
MACRO_TOL, N_BOOT = 0.0015, 2000
RARE3 = ['Pleural_Other', 'Fracture', 'Lung_Lesion']
WORST3_FIXED = ['Fracture', 'Pleural_Other', 'Enlarged_Cardiomediastinum']
BATCH = int(os.environ.get('LP1_BATCH', 64)); ACCUM = max(1, EFF // BATCH)

# ================= cong §14 =================
GATE = dict(macro_delta=0.002, rare3_delta=0.002, nontgt_delta=-0.0015, seeds_same_dir=2)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU'
print(f'device {dev} | {GPU_NAME}')
print(f'ARM {ARM_TAG} | SEED {SEED} | BATCH {BATCH} x ACCUM {ACCUM} = {BATCH*ACCUM}')
print(f'nhan  <- {LP1}')
print(f'H0    <- {SC16}  (chi doc)')
print(f'ra    -> {STU}')
print('\n!! Moi arm phai train tren CUNG mot GPU thi so sanh moi sach.')

## C1 · Cohort · ma trận nhãn của arm · VAL historical

`Y_HARD/M_HARD/Y_SOFT/M_SOFT` index bằng `study_row` toàn cohort nên cắm thẳng vào loader được.
VAL giữ **nhãn historical** (`+1/-1 → positive`, `0/blank → negative`) đúng plan §13.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; C = len(LABELS); assert C == 13
ehr = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'),
                       columns=['study_id', 'subject_id', 'split', 'setting'] + LABELS)
       .sort_values('study_id').reset_index(drop=True))
N = len(ehr)
Y_HIST    = ehr[LABELS].to_numpy('float32')          # U-Ones — dung cho VAL va cho H0
STUDY_ROW = {int(s): i for i, s in enumerate(ehr.study_id.to_numpy())}
SPLIT     = ehr.split.to_numpy()

cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
N_POS_TRAIN = np.asarray(cf['class_freq'], 'float64'); N_TR = int(cf['num_train'])
TIER = np.where(N_POS_TRAIN/N_TR < 0.02, 'tail', np.where(N_POS_TRAIN/N_TR < 0.10, 'medium', 'head'))
RARE3_IDX  = np.array([LABELS.index(l) for l in RARE3])
NONTGT_IDX = np.setdiff1d(np.arange(C), RARE3_IDX)
TAIL_IDX   = np.where(TIER == 'tail')[0]
W3_IDX     = np.array([LABELS.index(l) for l in WORST3_FIXED])

Y_HARD = np.load(f'{LP1}/Y_HARD.npy'); M_HARD = np.load(f'{LP1}/M_HARD.npy')
Y_SOFT = np.load(f'{LP1}/Y_SOFT.npy'); M_SOFT = np.load(f'{LP1}/M_SOFT.npy')
for a in (Y_HARD, M_HARD, Y_SOFT, M_SOFT): assert a.shape == (N, C)
assert not (M_HARD*M_SOFT).any(), 'mot cell vua hard vua soft'
_tr = SPLIT == 'train'
nh, ns = int(M_HARD[_tr].sum()), int(M_SOFT[_tr].sum())
nm = int(((M_HARD+M_SOFT)[_tr] == 0).sum())
print(f'ARM {ARM_TAG}: hard {nh:,} | soft {ns:,} | mask {nm:,} | tong {nh+ns+nm:,} (= {_tr.sum()*C:,})')
print(f'  soft_frac {ns/(_tr.sum()*C):.2%} | grad share soft (lambda {LAMBDA_SOFT}) '
      f'{LAMBDA_SOFT*ns/(LAMBDA_SOFT*ns+nh):.2%}')
EXP = json.load(open(f'{LP1}/experiment_manifest.json'))
print(f"  router: {EXP['blank_route_rule']} | teacher {EXP['teacher_tag']} ({EXP['teacher_mode']})")

# ---- view manifest ----
vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal', 'lateral'])]
_dup = [c for c in ('subject_id', 'split', 'setting') if c in vm.columns]
MAN_ALL = (vm.drop(columns=_dup).merge(ehr[['study_id', 'subject_id', 'split']], on='study_id',
                                       how='inner').sort_values('dicom_id').reset_index(drop=True))
print(f'anh: {dict(MAN_ALL.split.value_counts())}')
print('\nTEST khong duoc doc trong notebook nay.')

## C2 · Stage ảnh + dataset *(verbatim nb16)*

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
_have = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
if _have > 1000 and not os.path.isdir(SHARD_DIR):
    print(f'{_have:,} anh da co san; bo qua staging')
else:
    tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')) if os.path.isdir(SHARD_DIR) else []
    assert tars or _have > 1000, 'khong co shard va LOCAL_IMG rong'
    todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
    print(f'shards {len(tars)} | to stage {len(todo)} | da co {_have:,}')
    for f in tqdm(todo, desc='staging', unit='shard'):
        tmp = f'/content/{f}'
        shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
        with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
        os.remove(tmp); open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()
    print('local images:', len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')]))

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])

_before = len(MAN_ALL)
man = MAN_ALL[MAN_ALL.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))
              ].sort_values('dicom_id').reset_index(drop=True)
assert len(man) and (_before-len(man))/_before < 0.01, f'{_before-len(man)} anh thieu -> stage anh'
man['study_row'] = man.study_id.map(STUDY_ROW).astype('int64')
IDX = {s: np.where(man.split.to_numpy() == s)[0] for s in ('train', 'val', 'test')}
print(f'anh theo split: { {s: int(len(IDX[s])) for s in IDX} }')

class ImgDS(Dataset):
    '''Chi tra anh + study_row; target tra cuu tren GPU bang sr.'''
    def __init__(s, idx, tf): s.idx, s.tf = np.asarray(idx), tf
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        r = man.iloc[s.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))
        return dict(image=s.tf(im), sr=int(r.study_row))

def collate(bb):
    return dict(image=torch.stack([x['image'] for x in bb]),
                sr=torch.tensor([x['sr'] for x in bb], dtype=torch.int64))

def loader(idx, tf, shuffle, bs, workers=4):
    return DataLoader(ImgDS(idx, tf), batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev == 'cuda'), persistent_workers=(workers > 0))

to_dev = lambda b: {k: v.to(dev, non_blocking=True) for k, v in b.items()}
_b = to_dev(collate([ImgDS(IDX['val'], eval_tf)[k] for k in range(4)]))
print('dataset OK | image', tuple(_b['image'].shape))

## C3 · Model R0 + `student_loss`

In [ ]:
from transformers import AutoModel
TRAIN_FROM = 9
_probe = AutoModel.from_pretrained('microsoft/rad-dino')
N_BLOCKS = len(_probe.encoder.layer); D = _probe.config.hidden_size; del _probe
N_PATCH = (MODEL_SIZE // 14) ** 2
_sd04b = torch.load(BB_CKPT, map_location='cpu', weights_only=False)['model']
BB_SD = {k[9:]: v for k, v in _sd04b.items() if k.startswith('backbone.')}

def load_backbone(device=dev):
    m = AutoModel.from_pretrained('microsoft/rad-dino'); m.load_state_dict(BB_SD, strict=True)
    return m.to(device).eval()

backbone = load_backbone()
for p in backbone.parameters(): p.requires_grad_(False)
LAYERS = backbone.encoder.layer
def _layer(h, i):
    o = LAYERS[i](h); return o[0] if isinstance(o, (tuple, list)) else o

@torch.no_grad()
def forward_shared(x):
    h = backbone.embeddings(x)
    for i in range(TRAIN_FROM): h = _layer(h, i)
    return h.detach()

import sys, types
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched'):
    _oi = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw): _oi(self, *a, **kw); self.self_attn = types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__ = _init
    _of = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None); kw.pop('memory_is_causal', None); return _of(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd
    _mld.TransformerDecoderLayerOptimal._patched = True
H5_SD = torch.load(H5_CKPT, map_location='cpu', weights_only=False)['model']
_sub = lambda p: {k[len(p):]: v for k, v in H5_SD.items() if k.startswith(p)}
_TAIL_SRC = load_backbone('cpu')
for p in _TAIL_SRC.parameters(): p.requires_grad_(False)

class TailBackbone(nn.Module):
    def __init__(s):
        super().__init__()
        s.blocks = nn.ModuleList([copy.deepcopy(_TAIL_SRC.encoder.layer[i])
                                  for i in range(TRAIN_FROM, N_BLOCKS)])
        s.ln = copy.deepcopy(_TAIL_SRC.layernorm)
        for p in s.parameters(): p.requires_grad_(True)
    def forward(s, h):
        for b in s.blocks:
            o = b(h); h = o[0] if isinstance(o, (tuple, list)) else o
        return s.ln(h)

class H5Head(nn.Module):
    def __init__(s):
        super().__init__()
        s.norm = nn.LayerNorm(D)
        s.mld = MLDecoder(num_classes=C, initial_num_features=D, decoder_embedding=D, num_of_groups=C)
        s.norm.load_state_dict(_sub('norm.')); s.mld.load_state_dict(_sub('mld.'))
    def forward(s, patches): return s.mld(s.norm(patches))

class Arm(nn.Module):
    def __init__(s):
        super().__init__(); s.tail = TailBackbone(); s.head = H5Head()
    def forward(s, z):
        h = s.tail(z); assert h.shape[1] == N_PATCH + 1
        return s.head(h[:, 1:])

def build_arm(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    return Arm().to(dev)

tail_params = lambda m: list(m.tail.parameters())
head_params = lambda m: list(m.head.parameters())

class CellWASL(nn.Module):
    def __init__(s): super().__init__(); s.gn, s.gp, s.clip, s.eps = ASL_GN, ASL_GP, ASL_CLIP, ASL_EPS
    def forward(s, x, y, w):
        y = y.float(); xp = torch.sigmoid(x); xn = (1-xp+s.clip).clamp(max=1)
        lp = y*torch.log(xp.clamp(min=s.eps)); ln = (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y); mod = torch.pow(1-pt, g)
        return (w*(-lp*mod - ln*mod)).sum()/(x.shape[0]*x.shape[1])
CRIT = CellWASL().to(dev)

YH_t = torch.from_numpy(Y_HARD).to(dev); MH_t = torch.from_numpy(M_HARD).to(dev)
YS_t = torch.from_numpy(Y_SOFT).to(dev); MS_t = torch.from_numpy(M_SOFT).to(dev)
YHIST_t = torch.from_numpy(Y_HIST).to(dev)

def student_loss(z, sr, parts=False):
    yh, mh, ys, ms = YH_t[sr], MH_t[sr], YS_t[sr], MS_t[sr]
    hard = CRIT(z, yh, mh)
    soft = (ms*F.binary_cross_entropy_with_logits(z, ys, reduction='none')).sum()/(z.shape[0]*z.shape[1])
    tot = hard + LAMBDA_SOFT*soft
    return (tot, float(hard.detach()), float(soft.detach())) if parts else tot

def make_opt(m):
    def sp(ps, lr):
        d = [p for p in ps if p.ndim > 1]; nd = [p for p in ps if p.ndim <= 1]
        return [{'params': d, 'lr': lr, 'weight_decay': WD},
                {'params': nd, 'lr': lr, 'weight_decay': 0.0}]
    return torch.optim.AdamW(sp(tail_params(m), LR_BLOCKS) + sp(head_params(m), LR_HEAD))

set_head_trainable = lambda m, on: [p.requires_grad_(on) for p in head_params(m)]
def clip_arm(m):
    t = [p for p in tail_params(m) if p.grad is not None]
    h = [p for p in head_params(m) if p.grad is not None]
    return (float(nn.utils.clip_grad_norm_(t, CLIP_BLOCKS)) if t else 0.0,
            float(nn.utils.clip_grad_norm_(h, CLIP_HEAD)) if h else 0.0)

_A = build_arm(); _x = torch.randn(2, 3, MODEL_SIZE, MODEL_SIZE, device=dev)
with torch.no_grad():
    _d = (backbone(pixel_values=_x).last_hidden_state - _A.tail(forward_shared(_x))).abs().max().item()
print(f'G0 identity max|diff| {_d:.2e} {"OK" if _d < 1e-4 else "FAIL"}'); assert _d < 1e-4
_sr = torch.from_numpy(np.where(SPLIT == 'train')[0][:8]).to(dev)
_z = torch.randn(8, C, device=dev, requires_grad=True)
student_loss(_z, _sr).backward()
_mask = (M_HARD[_sr.cpu().numpy()] + M_SOFT[_sr.cpu().numpy()]) == 0
print(f'masked cell {int(_mask.sum())} | max|grad| tren masked '
      f'{np.abs(_z.grad.detach().cpu().numpy()[_mask]).max() if _mask.any() else 0:.2e}')
del _A; torch.cuda.empty_cache() if dev == 'cuda' else None
print('model + student_loss OK')

## C4 · Bộ đo + checkpoint rule §9 *(verbatim nb16)*

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, auc

def pool_study(p_img, split):
    sub = man.iloc[IDX[split]]
    d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id'] = sub.study_id.to_numpy(); d['vg'] = sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg == 'frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg == 'lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy().astype('float32'); hl = la.notna().all(1).to_numpy().astype('float32')
    den = (W_F*hf + W_L*hl)[:, None]
    return sid, (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:, None]
                 + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:, None]) / den

@torch.no_grad()
def predict(m, split, bs=None):
    m.eval(); bs = bs or BATCH
    dl = DataLoader(ImgDS(IDX[split], eval_tf), batch_size=bs, shuffle=False, collate_fn=collate,
                    num_workers=0, pin_memory=(dev == 'cuda'))
    P = np.zeros((len(IDX[split]), C), 'float32'); L = 0.0; nb = 0; pos = 0
    for b in tqdm(dl, desc=f'pred[{split}]', leave=False):
        b = to_dev(b); k = b['image'].shape[0]; z = m(forward_shared(b['image']))
        P[pos:pos+k] = torch.sigmoid(z).float().cpu().numpy(); pos += k
        L += float(CRIT(z, YHIST_t[b['sr']], torch.ones_like(z))); nb += 1
    return P, L/max(nb, 1)

def recall_at_k(y, s):
    K = int(y.sum())
    return float(y[np.argsort(-s, kind='stable')[:K]].sum())/K if K else np.nan

def metrics_from(Y, PS):
    ap = np.array([average_precision_score(Y[:, j], PS[:, j]) if Y[:, j].sum() else np.nan
                   for j in range(C)])
    auprc = np.full(C, np.nan); auroc = np.full(C, np.nan); rk = np.full(C, np.nan)
    for j in range(C):
        if Y[:, j].sum() == 0 or Y[:, j].sum() == len(Y): continue
        pr, rc, _ = precision_recall_curve(Y[:, j], PS[:, j])
        auprc[j] = auc(rc, pr); auroc[j] = roc_auc_score(Y[:, j], PS[:, j])
        rk[j] = recall_at_k(Y[:, j], PS[:, j])
    nm = lambda a, ix: float(np.nanmean(a[ix]))
    return dict(macro_map=float(np.nanmean(ap)), rare3_map=nm(ap, RARE3_IDX),
                nontgt_map=nm(ap, NONTGT_IDX), tail_map=nm(ap, TAIL_IDX),
                worst3_fixed=nm(ap, W3_IDX), tail_recall_k=nm(rk, TAIL_IDX),
                macro_auroc=float(np.nanmean(auroc)), macro_auprc=float(np.nanmean(auprc)),
                per_ap=ap.tolist())

SID_VAL, _ = pool_study(np.zeros((len(IDX['val']), C), 'float32'), 'val')
Y_VAL    = ehr.set_index('study_id').loc[SID_VAL, LABELS].to_numpy('float32')
SUBJ_VAL = ehr.set_index('study_id').loc[SID_VAL, 'subject_id'].to_numpy()
_p16 = f'{SC16}/preds/sid_val.npy'
if os.path.exists(_p16):
    assert (np.load(_p16) == SID_VAL).all(), 'SID_VAL lech nb16 -> khong so sanh duoc'
    print('SID_VAL khop nb16  OK')
print(f'VAL {len(SID_VAL):,} study | {len(np.unique(SUBJ_VAL)):,} subject | {len(IDX["val"]):,} anh')

def select_ckpt(log_arm):
    '''§9: Macro >= best - MACRO_TOL -> max Tail AP -> Worst-3 -> Recall@K -> Macro.'''
    e = {int(k): v for k, v in log_arm.items()}
    best = max(v['macro_map'] for v in e.values())
    cand = [k for k, v in e.items() if v['macro_map'] >= best - MACRO_TOL]
    return max(cand, key=lambda k: (e[k]['tail_map'], e[k]['worst3_fixed'],
                                    e[k]['tail_recall_k'], e[k]['macro_map']))
print('eval ready')

## C5 · Train — 8 epoch, eval VAL mỗi epoch, resume theo epoch

Chạy lại cell là tiếp tục đúng epoch đang dở. Mỗi epoch ghi `snapshots/ep{n}.pt`,
`preds/val_ep{n}.npy` và cập nhật `epoch_log.json`.

In [ ]:
LOG_PATH = f'{STU}/epoch_log.json'
resume_p = f'{STU}/_resume.pt'

def train_student():
    dl = loader(IDX['train'], train_tf, True, BATCH)
    STEPS = math.ceil(len(dl)/ACCUM); TOTAL = EPOCHS*STEPS; WARM = max(1, int(WARMUP_FRAC*TOTAL))
    def mk_sched(opt):
        cw = lambda s: s/WARM if s < WARM else 0.5*(1+math.cos(math.pi*(s-WARM)/max(TOTAL-WARM, 1)))
        return torch.optim.lr_scheduler.LambdaLR(opt, [cw, cw, lambda s: 1.0, lambda s: 1.0])
    LOG = json.load(open(LOG_PATH)) if os.path.exists(LOG_PATH) else {}
    m = build_arm(SEED); opt = make_opt(m); sch = mk_sched(opt); start = 0
    if os.path.exists(resume_p):
        ck = torch.load(resume_p, map_location=dev, weights_only=False)
        assert ck['batch'] == BATCH and ck['arm'] == ARM_TAG and ck['seed'] == SEED, \
            f'resume khac cau hinh ({ck["arm"]}/s{ck["seed"]}/B{ck["batch"]}) -> xoa {resume_p}'
        m.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
        start = ck['epoch'] + 1; print(f'RESUME @ep{start+1}')
        if start >= EPOCHS: print('da xong -> sang C6'); return LOG
    FROZEN0 = {k: v.detach().clone() for k, v in backbone.state_dict().items()}
    print(f'{len(IDX["train"]):,} anh | {STEPS:,} opt-step/epoch | total {TOTAL:,} | warmup {WARM}')
    for ep in range(start, EPOCHS):
        unf = ep >= HEAD_WARMUP_EPOCHS
        set_head_trainable(m, unf); m.train()
        torch.manual_seed(SEED*7919 + ep)
        acc = dict(l=0.0, h=0.0, s=0.0, n=0); t0 = time.time()
        for i, b in enumerate(tqdm(dl, desc=f's{SEED} ep{ep+1}/{EPOCHS}{"[all]" if unf else "[tail]"}',
                                   leave=False)):
            b = to_dev(b); z = m(forward_shared(b['image']))
            loss, lh, ls = student_loss(z, b['sr'], parts=True)
            assert torch.isfinite(loss), f'HARD FAILURE: loss non-finite ep{ep+1} step{i}'
            (loss/ACCUM).backward()
            acc['l'] += float(loss.detach()); acc['h'] += lh; acc['s'] += ls; acc['n'] += 1
            if ((i+1) % ACCUM == 0) or (i == len(dl)-1):
                nt, nh_ = clip_arm(m)
                assert math.isfinite(nt) and math.isfinite(nh_), 'HARD FAILURE: grad norm'
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        h_tr = (time.time()-t0)/3600
        assert not [k for k, v in backbone.state_dict().items() if not torch.equal(FROZEN0[k], v)], \
            'HARD FAILURE: blocks 0-8 bi doi'
        Pv, Lv = predict(m, 'val')
        sid, PS = pool_study(Pv, 'val'); assert (sid == SID_VAL).all()
        np.save(f'{STU}/preds/val_ep{ep+1}.npy', PS)
        r = metrics_from(Y_VAL, PS)
        n = max(acc['n'], 1)
        r['diag'] = dict(train_loss=acc['l']/n, train_hard=acc['h']/n, train_soft=acc['s']/n,
                         val_loss=Lv, hours=h_tr, head_unfrozen=bool(unf), gpu=GPU_NAME)
        LOG[str(ep+1)] = r
        json.dump(LOG, open(LOG_PATH, 'w'), indent=1)
        torch.save({k: v.detach().cpu() for k, v in m.state_dict().items()},
                   f'{STU}/snapshots/ep{ep+1}.pt')
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(),
                        epoch=ep, batch=BATCH, arm=ARM_TAG, seed=SEED), resume_p+'.tmp')
        os.replace(resume_p+'.tmp', resume_p)
        print(f'  ep{ep+1} {h_tr:.2f}h | Macro {r["macro_map"]:.4f} R3 {r["rare3_map"]:.4f} '
              f'NT {r["nontgt_map"]:.4f} tail {r["tail_map"]:.4f} w3 {r["worst3_fixed"]:.4f} '
              f'| L {r["diag"]["train_loss"]:.5f} (hard {r["diag"]["train_hard"]:.5f} '
              f'soft {r["diag"]["train_soft"]:.5f})  [saved]')
    print(f'\nDONE {ARM_TAG} seed {SEED} -> {LOG_PATH}')
    return LOG

LOG = train_student()

## C6 · So với `H0` + cổng §14

In [ ]:
LOG = json.load(open(LOG_PATH))
EP_A = select_ckpt(LOG); A = LOG[str(EP_A)]
print(f'=== {ARM_TAG} seed {SEED} — tung epoch ===')
T = pd.DataFrame([dict(ep=int(e), macro=v['macro_map'], rare3=v['rare3_map'], nontgt=v['nontgt_map'],
                       tail=v['tail_map'], worst3=v['worst3_fixed'],
                       train_loss=v['diag']['train_loss'], val_loss=v['diag']['val_loss'])
                  for e, v in sorted(LOG.items(), key=lambda x: int(x[0]))])
print(T.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'\ncheckpoint theo §9: ep{EP_A}')

# ---------------- H0 tu nb16 ----------------
H0_LOG = f'{SC16}/metrics/epoch_log_seed{SEED}.json'
if not os.path.exists(H0_LOG):
    print(f'\n!! khong thay {H0_LOG} -> chua co H0 cho seed {SEED}. '
          f'Seed 42 co san tu nb16; 1337/2024 phai train R0 truoc moi so sanh duoc.')
else:
    H = json.load(open(H0_LOG))['R0']
    EP_H = select_ckpt(H); B = H[str(EP_H)]
    print(f'\n=== H0 (R0 nb16, seed {SEED}) checkpoint ep{EP_H} ===')
    KEYS = ['macro_map', 'rare3_map', 'nontgt_map', 'tail_map', 'worst3_fixed']
    CMP = pd.DataFrame([dict(metric=k, H0=B[k], **{ARM_TAG: A[k]}, delta=A[k]-B[k]) for k in KEYS])
    print(CMP.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
    CMP.to_csv(f'{STU}/metrics/vs_H0.csv', index=False)

    dM, dR, dN = A['macro_map']-B['macro_map'], A['rare3_map']-B['rare3_map'], \
                 A['nontgt_map']-B['nontgt_map']
    g1 = (dM >= GATE['macro_delta']) or (dR >= GATE['rare3_delta'])
    g2 = dN >= GATE['nontgt_delta']
    print(f'\n=== CONG §14 (seed {SEED}) ===')
    print(f"  Macro delta {dM:+.4f} >= {GATE['macro_delta']:+.4f} ? {dM >= GATE['macro_delta']}")
    print(f"  Rare3 delta {dR:+.4f} >= {GATE['rare3_delta']:+.4f} ? {dR >= GATE['rare3_delta']}")
    print(f"  -> dieu kien 1 (Macro HOAC Rare3): {'PASS' if g1 else 'FAIL'}")
    print(f"  NonTgt delta {dN:+.4f} >= {GATE['nontgt_delta']:+.4f} ? {'PASS' if g2 else 'FAIL'}")
    print(f"  => seed {SEED}: {'PASS' if (g1 and g2) else 'FAIL'}")
    print(f"  (con can >= {GATE['seeds_same_dir']}/3 seed cung chieu — chay tiep 1337 / 2024)")

    print('\n=== per-label AP ===')
    PL = pd.DataFrame(dict(label=LABELS, H0=B['per_ap'], arm=A['per_ap']))
    PL['delta'] = PL.arm - PL.H0
    print(PL.sort_values('delta').to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
    PL.to_csv(f'{STU}/metrics/per_label_vs_H0.csv', index=False)
    if B['diag'].get('gpu') and B['diag']['gpu'] != GPU_NAME:
        print(f"\n!! H0 train tren {B['diag']['gpu']}, arm nay tren {GPU_NAME} — "
              f"khac phan cung, delta co the lan nhieu phan cung.")

## C7 · Bootstrap subject-cluster có ghép cặp *(plan §13 — 2,000 lần)*

Resample **subject** (không phải study) để tôn trọng tương quan trong cùng bệnh nhân. Ghép cặp:
mỗi lần lấy mẫu tính delta trên **cùng** tập subject cho cả hai arm.

In [ ]:
_pH = f'{SC16}/preds/R0_s{SEED}_ep{EP_H}.npy' if 'EP_H' in dir() else None
if not (_pH and os.path.exists(_pH)):
    print(f'khong thay prediction cua H0 ({_pH}) -> bo qua bootstrap')
else:
    PS_A = np.load(f'{STU}/preds/val_ep{EP_A}.npy'); PS_H = np.load(_pH)
    assert PS_A.shape == PS_H.shape == (len(SID_VAL), C)
    su, inv = np.unique(SUBJ_VAL, return_inverse=True)
    grp = [np.where(inv == k)[0] for k in range(len(su))]
    rng = np.random.default_rng(SEED)
    GRPS = dict(macro=np.arange(C), rare3=RARE3_IDX, nontgt=NONTGT_IDX, tail=TAIL_IDX)
    def _m(Y, P, ix):
        ap = [average_precision_score(Y[:, j], P[:, j]) if Y[:, j].sum() else np.nan for j in ix]
        return float(np.nanmean(ap))
    boot = {k: [] for k in GRPS}
    for _ in tqdm(range(N_BOOT), desc='bootstrap'):
        pick = np.concatenate([grp[k] for k in rng.integers(0, len(su), len(su))])
        Yb = Y_VAL[pick]
        for k, ix in GRPS.items():
            ok = [j for j in ix if Yb[:, j].sum() > 0]
            if not ok: boot[k].append(np.nan); continue
            boot[k].append(_m(Yb, PS_A[pick], ok) - _m(Yb, PS_H[pick], ok))
    print(f'\n=== DELTA {ARM_TAG} - H0, bootstrap {N_BOOT} lan (cluster = subject) ===')
    rows = []
    for k, v in boot.items():
        v = np.asarray(v, 'float64'); v = v[np.isfinite(v)]
        lo, hi = np.percentile(v, [2.5, 97.5])
        rows.append(dict(metric=k, delta_mean=v.mean(), ci_lo=lo, ci_hi=hi,
                         p_gt_0=float((v > 0).mean()), harm=bool(hi < 0)))
    BT = pd.DataFrame(rows); BT.to_csv(f'{STU}/metrics/bootstrap_vs_H0.csv', index=False)
    print(BT.to_string(index=False, float_format=lambda v: f'{v:+.5f}'))
    print('\n  harm=True nghia la CI95 nam hoan toan duoi 0 -> co hai that su, khong phai nhieu.')
    json.dump(dict(arm=ARM_TAG, seed=SEED, ep_arm=EP_A, ep_h0=EP_H, gpu=GPU_NAME,
                   n_boot=N_BOOT, rows=rows), open(f'{STU}/metrics/summary.json', 'w'), indent=1)
    print(f'\n-> {STU}/metrics/')

## C8 · **Chẩn đoán** — known-only + bảng cơ chế

`known-only` là **diagnostic**, không thay primary endpoint (nb16 đã prereg như vậy). Checkpoint
lấy đúng cái rule §9 chọn trên metric chính — **không** chọn lại theo known-only.

```
(B) chinh   bo o raw == -1 cua tung nhan; giu +1 = pos, 0/blank = neg
            -> bo dung ~2.5% o VAL ma thuoc do va can thiep bat dong
            -> prevalence 0.9-33%, cung che do voi historical
(A) phu     chi giu +1 / 0. Prevalence 19-100% -> doc bang AUROC,
            bo nhan suy bien (n_neg == 0 hoac prevalence > 0.93)
```

Bảng cơ chế trả lời câu quan trọng nhất: **model có thật sự học cái ta dạy không** — điểm trung
bình trên ô `-1` của arm phải thấp hơn `H0` nếu can thiệp tới được model.

In [ ]:
BLANK_, NEG_, UNC_, POS_ = 0, 1, 2, 3
_raw = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_raw_labels.parquet'))
_R = (_raw.drop_duplicates('study_id').set_index('study_id')
      .reindex(SID_VAL)[[f'raw_{l}' for l in LABELS]].to_numpy('float64'))
ST_VAL = np.full((len(SID_VAL), C), BLANK_, 'int8')
ST_VAL[_R == 0] = NEG_; ST_VAL[_R == -1] = UNC_; ST_VAL[_R == 1] = POS_
assert ((ST_VAL == POS_) | (ST_VAL == UNC_)).astype('float32').sum() == Y_VAL.sum(), \
    'raw VAL khong tai tao duoc Y_VAL (U-Ones)'
print('raw VAL OK | trang thai:',
      {k: int((ST_VAL == v).sum()) for k, v in [('+1', POS_), ('0', NEG_),
                                                ('-1', UNC_), ('blank', BLANK_)]})

# mask KHONG phu thuoc arm -> hai arm duoc cham tren CUNG tap o
MASK_B = ST_VAL != UNC_                                   # bo o tranh chap
MASK_A = (ST_VAL == POS_) | (ST_VAL == NEG_)              # chi o dut khoat
Y_KNOWN = (ST_VAL == POS_).astype('float32')

def masked_metrics(P, mask):
    ap = np.full(C, np.nan); au = np.full(C, np.nan)
    npos = np.zeros(C, 'int64'); nneg = np.zeros(C, 'int64')
    for j in range(C):
        m = mask[:, j]; y = Y_KNOWN[m, j]; s_ = P[m, j]
        npos[j] = int(y.sum()); nneg[j] = int(len(y) - y.sum())
        if npos[j] == 0 or nneg[j] == 0: continue
        ap[j] = average_precision_score(y, s_); au[j] = roc_auc_score(y, s_)
    nm = lambda a, ix: float(np.nanmean(a[ix])) if np.isfinite(a[ix]).any() else np.nan
    return dict(per_ap=ap, per_auroc=au, n_pos=npos, n_neg=nneg,
                macro_ap=float(np.nanmean(ap)), rare3_ap=nm(ap, RARE3_IDX),
                nontgt_ap=nm(ap, NONTGT_IDX), macro_auroc=float(np.nanmean(au)))

_pH = f'{SC16}/preds/R0_s{SEED}_ep{EP_H}.npy'
assert os.path.exists(f'{STU}/preds/val_ep{EP_A}.npy') and os.path.exists(_pH), \
    'thieu prediction -> chay C5/C6 truoc'
PS_A = np.load(f'{STU}/preds/val_ep{EP_A}.npy'); PS_H = np.load(_pH)
print(f'checkpoint (rule §9 tren metric CHINH): {ARM_TAG} ep{EP_A} | H0 ep{EP_H}')

KO = {}
for tag, mask, note in [('B', MASK_B, 'chan doan CHINH — bo o -1'),
                        ('A', MASK_A, 'phu — chi +1/0, doc AUROC')]:
    mA = masked_metrics(PS_A, mask); mH = masked_metrics(PS_H, mask); KO[tag] = (mA, mH)
    tot = (mA['n_pos'] + mA['n_neg']).clip(min=1)
    dead = [LABELS[j] for j in range(C) if mA['n_neg'][j] == 0 or mA['n_pos'][j] == 0
            or mA['n_pos'][j]/tot[j] > 0.93]
    print(f'\n=== KNOWN-ONLY ({tag}) — {note} ===')
    print(f'  o dung: {int(mask.sum()):,} / {mask.size:,} ({mask.mean():.1%})')
    if dead: print(f'  !! nhan suy bien (thieu mot lop hoac prevalence >93%): {dead}')
    D = pd.DataFrame(dict(label=LABELS, n_pos=mA['n_pos'], n_neg=mA['n_neg'],
                          prev=mA['n_pos']/tot, AP_H0=mH['per_ap'], AP_arm=mA['per_ap'],
                          dAP=mA['per_ap']-mH['per_ap'],
                          dAUROC=mA['per_auroc']-mH['per_auroc']))
    print(D.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
    for k in ('macro_ap', 'rare3_ap', 'nontgt_ap', 'macro_auroc'):
        print(f'  {k:<12} H0 {mH[k]:.4f} | {ARM_TAG} {mA[k]:.4f} | delta {mA[k]-mH[k]:+.4f}')
    D.to_csv(f'{STU}/metrics/known_only_{tag}.csv', index=False)

# ---------------- BANG CO CHE ----------------
print('\n=== CO CHE: diem du doan trung binh theo TRANG THAI THO ===')
rows = []
for j, l in enumerate(LABELS):
    r = dict(label=l, n_unc=int((ST_VAL[:, j] == UNC_).sum()))
    for nm_, st in [('+1', POS_), ('0', NEG_), ('-1', UNC_), ('blank', BLANK_)]:
        m = ST_VAL[:, j] == st
        r[f'H0_{nm_}'] = float(PS_H[m, j].mean()) if m.any() else np.nan
        r[f'arm_{nm_}'] = float(PS_A[m, j].mean()) if m.any() else np.nan
    r['d_unc'] = r['arm_-1'] - r['H0_-1']
    rows.append(r)
MC = pd.DataFrame(rows); MC.to_csv(f'{STU}/metrics/mechanism_by_raw_state.csv', index=False)
print(MC[['label', 'n_unc', 'H0_-1', 'arm_-1', 'd_unc', 'H0_+1', 'arm_+1',
          'H0_blank', 'arm_blank']].to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
_mv = MC[MC.n_unc > 0]
print('\n  d_unc = diem tren o `-1` cua arm TRU cua H0.')
print('    am ro  -> can thiep DA toi duoc model (dung thiet ke)')
print('    ~0     -> 33,201 o soft khong uon duoc model; tranh luan ve metric la thua')
print(f'  hien tai: trung binh {_mv.d_unc.mean():+.4f} | '
      f'am o {int((_mv.d_unc < 0).sum())}/{len(_mv)} nhan')

### C8b · Bootstrap known-only (B) + đường cong theo epoch *(minh bạch, không dùng để chọn)*

In [ ]:
su, inv = np.unique(SUBJ_VAL, return_inverse=True)
grp = [np.where(inv == k)[0] for k in range(len(su))]
rng = np.random.default_rng(SEED)
GRPS = dict(macro=np.arange(C), rare3=RARE3_IDX, nontgt=NONTGT_IDX)
boot = {k: [] for k in GRPS}
for _ in tqdm(range(N_BOOT), desc='bootstrap known-only'):
    pick = np.concatenate([grp[k] for k in rng.integers(0, len(su), len(su))])
    yb = Y_KNOWN[pick]; mb = MASK_B[pick]; pa = PS_A[pick]; ph = PS_H[pick]
    for k, ix in GRPS.items():
        da = []
        for j in ix:
            m = mb[:, j]; y = yb[m, j]
            if y.sum() == 0 or y.sum() == len(y): continue
            da.append(average_precision_score(y, pa[m, j])
                      - average_precision_score(y, ph[m, j]))
        boot[k].append(float(np.mean(da)) if da else np.nan)
print(f'\n=== DELTA known-only(B): {ARM_TAG} - H0 | bootstrap {N_BOOT} (cluster = subject) ===')
rows = []
for k, v in boot.items():
    v = np.asarray(v, 'float64'); v = v[np.isfinite(v)]
    lo, hi = np.percentile(v, [2.5, 97.5])
    rows.append(dict(metric=k, delta_mean=v.mean(), ci_lo=lo, ci_hi=hi,
                     p_gt_0=float((v > 0).mean()), help=bool(lo > 0), harm=bool(hi < 0)))
BK = pd.DataFrame(rows); BK.to_csv(f'{STU}/metrics/bootstrap_known_only.csv', index=False)
print(BK.to_string(index=False, float_format=lambda v: f'{v:+.5f}'))
print('  help=True: CI95 hoan toan tren 0 | harm=True: hoan toan duoi 0')

print('\n=== duong cong known-only(B) qua epoch (KHONG dung de chon checkpoint) ===')
cur = []
for e in sorted(int(x) for x in LOG):
    f_ = f'{STU}/preds/val_ep{e}.npy'
    if not os.path.exists(f_): continue
    m_ = masked_metrics(np.load(f_), MASK_B)
    cur.append(dict(ep=e, macro_hist=LOG[str(e)]['macro_map'], macro_known=m_['macro_ap'],
                    rare3_known=m_['rare3_ap'], auroc_known=m_['macro_auroc'],
                    chon='<- §9' if e == EP_A else ''))
print(pd.DataFrame(cur).to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'\n-> {STU}/metrics/')